# 1_2_1 PageRank 与 MetaPath 构建

本 Notebook 由 [`3_0_2 Retevie.ipynb`](./3_0_2%20Retevie.ipynb) 拆分迁移：

- **E** 分子图 GDS 投影 + 中心性写回（节点/关系对齐 schema）
- **F** MetaPath 构建（F1–F4）

**数据源**：

| 文件 | 用途 |
|------|------|
| `output/potential_schema.json` | F1 手写清单的对照来源 |
| `output/subgraph_mapping.json` | E 段 GDS 节点列表 + 子图归属 |
| `output/relation.json` | 过滤 DEPRECATED 关系类型 |

**执行顺序**：E → F1 清单/验收 → **F1 low 构建** → **F4 mid + 层级边** → **验收** → F2 → F3

**前置**：KG 已由 [`1_2_0_2build_kg__neo4j.ipynb`](./1_2_0_2build_kg__neo4j.ipynb) 构建（模块4 子图标注 + 模块5 chunk 去重）。

**下游**：[`3_0_2 Retevie.ipynb`](./3_0_2%20Retevie.ipynb) G 段对话检索。


# E 分子图图分析并赋值

对 MPU / EBM / EEM 三子图分别做 GDS 投影，计算中心性并写回 `{sg}_pagerank`。

- **节点**：`subgraph_mapping.json` v1.2（手写 `SUBGRAPH_CONFIGS`）
- **关系**：与 F1 手写清单一致的非 F4 关系类型
- **下游**：F1 low / F4 mid 的 `maxPageRank` **直接读取**本段写回的 `{mpu,eem,ebm}_pagerank`；未执行 E 则 F 段将 raise


In [1]:
# ══════════════════════════════════════════════════════════════
# 三子图中心性计算与写回（SUBGRAPH_CONFIGS 对齐 subgraph_mapping v1.2）
# ══════════════════════════════════════════════════════════════

from graphdatascience import GraphDataScience
import pandas as pd

SUBGRAPH_CONFIGS = {'mpu': {'projection': 'mpu_projection',
         'nodes': ['whu_Goal',
                   'mp_References',
                   'whu_DataSet',
                   'mp_Claim',
                   'mp_Statement',
                   'whu_Method',
                   'mp_Attribution',
                   'whu_ScienceEvidence',
                   'whu_SupportGraph'],
         'relationships': {'MP_SUPPORTS': {'type': 'mp_supports', 'orientation': 'NATURAL'},
                           'MP_CHALLENGES': {'type': 'mp_challenges', 'orientation': 'NATURAL'},
                           'CITO_ISCITEDBY': {'type': 'cito_isCitedBy', 'orientation': 'NATURAL'}}},
 'ebm': {'projection': 'ebm_projection',
         'nodes': ['whu_DataSet',
                   'whu_Specimen_ProcessingStep',
                   'whu_Specimen_CollectionStep',
                   'whu_ProcessedSpecimen',
                   'whu_ScalarMeasurementDatum',
                   'whu_Device',
                   'whu_Specimen',
                   'whu_EnvironmentFeature',
                   'whu_SpecimenCollection',
                   'whu_SpecimenPreprocessing',
                   'whu_Reagent',
                   'whu_Software',
                   'envo_Material',
                   'whu_BioChemicalStep',
                   'whu_ComputationalStep',
                   'whu_Bio_chemical_Experiment'],
         'relationships': {'WHU_HASCONTEXT': {'type': 'whu_hasContext', 'orientation': 'NATURAL'},
                           'WHU_FELLOW': {'type': 'whu_fellow', 'orientation': 'NATURAL'},
                           'PROV_WASDERIVEDFROM': {'type': 'prov_wasDerivedFrom',
                                                   'orientation': 'NATURAL'},
                           'WHU_ATLOCATION': {'type': 'whu_atLocation', 'orientation': 'NATURAL'},
                           'WHU_DECLAREUSED': {'type': 'whu_declareUsed', 'orientation': 'NATURAL'},
                           'P_PLAN_HASOUTPUTVAR': {'type': 'p_plan_hasOutputVar',
                                                   'orientation': 'NATURAL'},
                           'P_PLAN_ISPRECEDEDBY': {'type': 'p_plan_isPrecededBy',
                                                   'orientation': 'NATURAL'},
                           'P_PLAN_HASINPUTVAR': {'type': 'p_plan_hasInputVar',
                                                  'orientation': 'NATURAL'},
                           'DCTERMS_HASPART': {'type': 'dcterms_hasPart', 'orientation': 'NATURAL'},
                           'IAO_IS_ABOUT': {'type': 'iao_is_about', 'orientation': 'NATURAL'}}},
 'eem': {'projection': 'eem_projection',
         'nodes': ['whu_DataSet',
                   'whu_BioChemicalStep',
                   'whu_Method',
                   'whu_ComputationalStep',
                   'whu_ProcessedSpecimen',
                   'whu_ScalarMeasurementDatum',
                   'whu_Device',
                   'whu_Bio_chemical_Experiment',
                   'whu_Specimen',
                   'whu_Computational_Experiment',
                   'whu_Goal',
                   'whu_Reagent',
                   'whu_Target_analyte',
                   'whu_Software',
                   'envo_Material',
                   'whu_SupportGraph',
                   'whu_ScienceEvidence',
                   'whu_Specimen_CollectionStep',
                   'whu_Specimen_ProcessingStep'],
         'relationships': {'WHU_FELLOW': {'type': 'whu_fellow', 'orientation': 'NATURAL'},
                           'WHU_HASGOAL': {'type': 'whu_hasGoal', 'orientation': 'NATURAL'},
                           'WHU_TARGET': {'type': 'whu_target', 'orientation': 'NATURAL'},
                           'PROV_WASDERIVEDFROM': {'type': 'prov_wasDerivedFrom',
                                                   'orientation': 'NATURAL'},
                           'WHU_DECLAREUSED': {'type': 'whu_declareUsed', 'orientation': 'NATURAL'},
                           'P_PLAN_HASINPUTVAR': {'type': 'p_plan_hasInputVar',
                                                  'orientation': 'NATURAL'},
                           'P_PLAN_HASOUTPUTVAR': {'type': 'p_plan_hasOutputVar',
                                                   'orientation': 'NATURAL'},
                           'P_PLAN_ISPRECEDEDBY': {'type': 'p_plan_isPrecededBy',
                                                   'orientation': 'NATURAL'}}}}

url = "bolt://localhost:7687"
username = "neo4j"
password = "tomis1cat"
gds = GraphDataScience(url, auth=(username, password))

print("SUBGRAPH_CONFIGS:")
for sg, cfg in SUBGRAPH_CONFIGS.items():
    print(f"  {sg.upper()}: nodes={len(cfg['nodes'])}, rel_types={len(cfg['relationships'])}")


def _drop_if_exists(proj_name: str):
    try:
        G = gds.graph.get(proj_name)
        gds.graph.drop(G)
        print(f"  已释放旧投影: {proj_name}")
    except Exception:
        pass


def _compute_and_write(G, sg: str):
    prefix = sg

    r = gds.degree.write(G, writeProperty=f"{prefix}_degree")
    print(f"  [{sg}] degree    written: {r['nodePropertiesWritten']}")

    r = gds.pageRank.write(
        G,
        writeProperty=f"{prefix}_pagerank",
        maxIterations=20,
        dampingFactor=0.85,
    )
    print(
        f"  [{sg}] pagerank  written: {r['nodePropertiesWritten']} "
        f"| converged: {r['didConverge']}"
    )

    r = gds.betweenness.write(G, writeProperty=f"{prefix}_betweenness")
    print(f"  [{sg}] betweenness written: {r['nodePropertiesWritten']}")

    r = gds.closeness.write(G, writeProperty=f"{prefix}_closeness")
    print(f"  [{sg}] closeness written: {r['nodePropertiesWritten']}")

    node_labels = SUBGRAPH_CONFIGS[sg]["nodes"]
    label_filter = " OR ".join(f"n:{lbl}" for lbl in node_labels)

    df = gds.run_cypher(f"""
        MATCH (n)
        WHERE ({label_filter})
          AND n.{prefix}_pagerank IS NOT NULL
        RETURN id(n)               AS nodeId,
               n.{prefix}_degree      AS degree,
               n.{prefix}_pagerank    AS pagerank,
               n.{prefix}_betweenness AS betweenness,
               n.{prefix}_closeness   AS closeness
    """)

    if df.empty:
        print(f"  [{sg}] ⚠️  无节点，跳过综合得分")
        return

    for col in ["degree", "pagerank", "betweenness", "closeness"]:
        mn, mx = df[col].min(), df[col].max()
        df[f"{col}_norm"] = (df[col] - mn) / (mx - mn + 1e-10)

    df[f"{prefix}_combined"] = df[
        ["degree_norm", "pagerank_norm", "betweenness_norm", "closeness_norm"]
    ].mean(axis=1)

    records = df[["nodeId", f"{prefix}_combined"]].to_dict("records")
    gds.run_cypher(
        f"""
        UNWIND $rows AS row
        MATCH (n) WHERE id(n) = row.nodeId
        SET n.{prefix}_combined = row.{prefix}_combined
    """,
        params={"rows": records},
    )

    print(f"  [{sg}] combined  written: {len(df)}")

    top3 = df.nlargest(3, f"{prefix}_combined")[
        ["nodeId", f"{prefix}_combined", "pagerank"]
    ]
    print(f"  [{sg}] Top 3 combined:\n{top3.to_string(index=False)}\n")


for sg, cfg in SUBGRAPH_CONFIGS.items():
    proj_name = cfg["projection"]
    print(f"\n{'=' * 60}")
    print(f"处理子图: {sg.upper()}  →  投影: {proj_name}")
    print("=" * 60)

    _drop_if_exists(proj_name)

    G, result = gds.graph.project(
        proj_name,
        node_spec=cfg["nodes"],
        relationship_spec=cfg["relationships"],
    )
    print(f"  投影建立: nodes={result['nodeCount']}, rels={result['relationshipCount']}")

    _compute_and_write(G, sg)

    gds.graph.drop(G)
    print(f"  投影已释放: {proj_name}")

print("\n✅ 全部三个子图中心性计算完成")

# ── E 段验收：Step 节点 pagerank 抽样 ─────────────────────────
print("\n" + "=" * 60)
print("E 段验收：Step 类节点 pagerank 抽样")
print("=" * 60)

STEP_CHECKS = [
    ("whu_BioChemicalStep", "eem_pagerank"),
    ("whu_Specimen_CollectionStep", "ebm_pagerank"),
    ("mp_Claim", "mpu_pagerank"),
]

for label, prop in STEP_CHECKS:
    sample = gds.run_cypher(f"""
        MATCH (n:{label})
        WHERE n.{prop} IS NOT NULL
        RETURN count(n) AS cnt,
               avg(n.{prop}) AS avg_pr,
               max(n.{prop}) AS max_pr
    """)
    row = sample.iloc[0]
    status = "✅" if row["cnt"] > 0 else "⚠️"
    print(f"  {status} {label} / {prop}: count={int(row['cnt'])}, avg={row['avg_pr']:.6f}, max={row['max_pr']:.6f}")

print("\n" + "=" * 60)
print("验证：各子图 Top 3 综合得分节点")
print("=" * 60)

for sg in ["mpu", "eem", "ebm"]:
    node_labels = SUBGRAPH_CONFIGS[sg]["nodes"]
    label_filter = " OR ".join(f"n:{lbl}" for lbl in node_labels)
    sample = gds.run_cypher(f"""
        MATCH (n)
        WHERE ({label_filter})
          AND n.{sg}_combined IS NOT NULL
        RETURN labels(n)        AS labels,
               n.WHU_HASNAME    AS name,
               n.{sg}_pagerank  AS pagerank,
               n.{sg}_combined  AS combined
        ORDER BY n.{sg}_combined DESC
        LIMIT 3
    """)
    print(f"\n[{sg.upper()}]")
    print(sample.to_string(index=False))


SUBGRAPH_CONFIGS:
  MPU: nodes=9, rel_types=3
  EBM: nodes=16, rel_types=10
  EEM: nodes=19, rel_types=8

处理子图: MPU  →  投影: mpu_projection
  投影建立: nodes=6576, rels=1182
  [mpu] degree    written: 6576
  [mpu] pagerank  written: 6576 | converged: True
  [mpu] betweenness written: 6576
  [mpu] closeness written: 6576
  [mpu] combined  written: 6576
  [mpu] Top 3 combined:
 nodeId  mpu_combined  pagerank
  12287      0.596429    0.4050
  12289      0.585714    0.2775
  18415      0.585714    0.2775

  投影已释放: mpu_projection

处理子图: EBM  →  投影: ebm_projection
  投影建立: nodes=2554, rels=561
  [ebm] degree    written: 2554
  [ebm] pagerank  written: 2554 | converged: True
  [ebm] betweenness written: 2554
  [ebm] closeness written: 2554
  [ebm] combined  written: 2554
  [ebm] Top 3 combined:
 nodeId  ebm_combined  pagerank
   9985      0.669318  1.112625
   9630      0.535633  0.934423
   9984      0.533819  1.095731

  投影已释放: ebm_projection

处理子图: EEM  →  投影: eem_projection
  投影建立: nodes=3300, 

# F1 构建meta path node

In [ ]:
# 清理 MetaPath Node 和 metaPathRelation 关系
with neo4j_driver.session() as session:
    result = session.run("""
        MATCH (mp:MetaPath)
        DETACH DELETE mp
        RETURN count(mp) AS deleted
    """).single()
    print(f"✅ 删除 MetaPath Node: {result['deleted']} 条")

# 删除索引
from neo4j_graphrag.indexes import drop_index_if_exists

drop_index_if_exists(neo4j_driver, "metapath_embedding_index")
drop_index_if_exists(neo4j_driver, "metapath_fulltext_index")
print("✅ 索引已删除")

## F1.1 关系清单（手写，对齐 potential_schema.json v1.2）

1. 下方 `SUBGRAPH_RELATIONS` 为手写定义（方向 `source -[relation]-> target`）
2. 不含 F4 层级边（见 `F4_RELATIONS`）
3. 运行验收 cell 确认 Neo4j 匹配后，再执行 `build_all_metapaths(neo4j_driver)`


In [ ]:
! pip install graphdatascience

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

neo4j_driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)

print("✅ Neo4j 连接已建立")

In [1]:
# ══════════════════════════════════════════════════════════════
# F1 关系清单（手写，对齐 potential_schema.json v1.2）
# 方向：source -[relation]-> target
# 不含 F4：p_plan_isStepOfPlan / whu_hasPart（见 F4_RELATIONS）
# ══════════════════════════════════════════════════════════════

SUBGRAPH_RELATIONS = {
    "MPU": [
        # mp_supports (12)
        ("mp_Attribution", "mp_supports", "whu_DataSet"),
        ("mp_Attribution", "mp_supports", "whu_Method"),
        ("mp_Attribution", "mp_supports", "mp_Claim"),
        ("mp_Attribution", "mp_supports", "mp_References"),
        ("mp_Attribution", "mp_supports", "mp_Statement"),
        ("whu_DataSet", "mp_supports", "mp_Claim"),
        ("whu_DataSet", "mp_supports", "mp_Statement"),
        ("whu_Method", "mp_supports", "whu_DataSet"),
        ("mp_Statement", "mp_supports", "mp_Claim"),
        ("mp_Claim", "mp_supports", "mp_Claim"),
        ("whu_ScienceEvidence", "mp_supports", "whu_SupportGraph"),
        ("whu_SupportGraph", "mp_supports", "mp_Claim"),
        # mp_challenges (6)
        ("whu_DataSet", "mp_challenges", "mp_Claim"),
        ("whu_DataSet", "mp_challenges", "mp_Statement"),
        ("mp_Statement", "mp_challenges", "mp_Claim"),
        ("mp_Claim", "mp_challenges", "mp_Claim"),
        ("whu_ScienceEvidence", "mp_challenges", "whu_SupportGraph"),
        ("whu_SupportGraph", "mp_challenges", "mp_Claim"),
        # cito_isCitedBy (4)
        ("mp_References", "cito_isCitedBy", "whu_DataSet"),
        ("mp_References", "cito_isCitedBy", "whu_Method"),
        ("mp_References", "cito_isCitedBy", "mp_Claim"),
        ("mp_References", "cito_isCitedBy", "mp_Statement"),
    ],
    "EBM": [
        ("whu_SpecimenCollection", "whu_hasContext", "whu_EnvironmentFeature"),
        ("whu_SpecimenPreprocessing", "whu_fellow", "whu_SpecimenCollection"),
        ("whu_Specimen", "prov_wasDerivedFrom", "whu_EnvironmentFeature"),
        ("whu_ProcessedSpecimen", "prov_wasDerivedFrom", "whu_Specimen"),
        ("whu_Bio_chemical_Experiment", "whu_fellow", "whu_SpecimenPreprocessing"),
        ("whu_Specimen_CollectionStep", "whu_atLocation", "whu_EnvironmentFeature"),
        ("whu_Specimen_CollectionStep", "whu_declareUsed", "whu_Method"),
        ("whu_Specimen_CollectionStep", "whu_declareUsed", "whu_Device"),
        ("whu_Specimen_CollectionStep", "whu_declareUsed", "envo_Material"),
        ("whu_Specimen_CollectionStep", "p_plan_hasOutputVar", "whu_Specimen"),
        ("whu_Specimen_CollectionStep", "p_plan_isPrecededBy", "whu_Specimen_CollectionStep"),
        ("whu_Specimen_ProcessingStep", "whu_declareUsed", "whu_Method"),
        ("whu_Specimen_ProcessingStep", "whu_declareUsed", "whu_Device"),
        ("whu_Specimen_ProcessingStep", "whu_declareUsed", "envo_Material"),
        ("whu_Specimen_ProcessingStep", "p_plan_hasInputVar", "whu_Specimen"),
        ("whu_Specimen_ProcessingStep", "p_plan_hasOutputVar", "whu_ProcessedSpecimen"),
        ("whu_Specimen_ProcessingStep", "p_plan_isPrecededBy", "whu_Specimen_CollectionStep"),
        ("whu_Specimen_ProcessingStep", "p_plan_isPrecededBy", "whu_Specimen_ProcessingStep"),
        ("whu_BioChemicalStep", "p_plan_isPrecededBy", "whu_Specimen_ProcessingStep"),
        ("whu_DataSet", "dcterms_hasPart", "whu_ScalarMeasurementDatum"),
        ("whu_DataSet", "iao_is_about", "whu_Reagent"),
        ("whu_DataSet", "iao_is_about", "whu_Specimen"),
        ("whu_DataSet", "iao_is_about", "whu_ProcessedSpecimen"),
    ],
    "EEM": [
        ("whu_Bio_chemical_Experiment", "whu_fellow", "whu_Bio_chemical_Experiment"),
        ("whu_Bio_chemical_Experiment", "whu_hasGoal", "whu_Goal"),
        ("whu_Computational_Experiment", "whu_fellow", "whu_Bio_chemical_Experiment"),
        ("whu_Computational_Experiment", "whu_fellow", "whu_Computational_Experiment"),
        ("whu_Computational_Experiment", "whu_hasGoal", "whu_Goal"),
        ("whu_Bio_chemical_Experiment", "whu_fellow", "whu_SpecimenPreprocessing"),
        ("whu_Goal", "whu_target", "whu_Target_analyte"),
        ("whu_ScienceEvidence", "prov_wasDerivedFrom", "whu_Computational_Experiment"),
        ("whu_BioChemicalStep", "whu_declareUsed", "whu_Method"),
        ("whu_BioChemicalStep", "whu_declareUsed", "whu_Device"),
        ("whu_BioChemicalStep", "whu_declareUsed", "whu_Reagent"),
        ("whu_BioChemicalStep", "p_plan_hasInputVar", "whu_ProcessedSpecimen"),
        ("whu_BioChemicalStep", "p_plan_hasInputVar", "whu_DataSet"),
        ("whu_BioChemicalStep", "p_plan_hasOutputVar", "whu_DataSet"),
        ("whu_BioChemicalStep", "p_plan_hasOutputVar", "whu_ProcessedSpecimen"),
        ("whu_BioChemicalStep", "p_plan_isPrecededBy", "whu_BioChemicalStep"),
        ("whu_ComputationalStep", "whu_declareUsed", "whu_Method"),
        ("whu_ComputationalStep", "whu_declareUsed", "whu_Software"),
        ("whu_ComputationalStep", "whu_declareUsed", "whu_DataSet"),
        ("whu_ComputationalStep", "p_plan_hasInputVar", "whu_DataSet"),
        ("whu_ComputationalStep", "p_plan_hasOutputVar", "whu_DataSet"),
        ("whu_ComputationalStep", "p_plan_isPrecededBy", "whu_BioChemicalStep"),
        ("whu_ComputationalStep", "p_plan_isPrecededBy", "whu_ComputationalStep"),
    ],
}

F4_RELATIONS = [
    ("whu_Specimen_CollectionStep", "p_plan_isStepOfPlan", "whu_SpecimenCollection"),
    ("whu_Specimen_ProcessingStep", "p_plan_isStepOfPlan", "whu_SpecimenPreprocessing"),
    ("whu_BioChemicalStep", "p_plan_isStepOfPlan", "whu_Bio_chemical_Experiment"),
    ("whu_ComputationalStep", "p_plan_isStepOfPlan", "whu_Computational_Experiment"),
    ("whu_ScienceEvidence", "whu_hasPart", "whu_DataSet"),
    ("whu_ScienceEvidence", "whu_hasPart", "whu_Method"),
    ("whu_SupportGraph", "whu_hasPart", "mp_Statement"),
    ("whu_SupportGraph", "whu_hasPart", "mp_Attribution"),
    ("whu_SupportGraph", "whu_hasPart", "mp_References"),
    ("whu_SupportGraph", "whu_hasPart", "whu_ScienceEvidence"),
]

PAGERANK_PROP = {
    "MPU": "mpu_pagerank",
    "EEM": "eem_pagerank",
    "EBM": "ebm_pagerank",
}

total = sum(len(v) for v in SUBGRAPH_RELATIONS.values())
print(f"✅ 关系清单：MPU={len(SUBGRAPH_RELATIONS['MPU'])}, "
      f"EBM={len(SUBGRAPH_RELATIONS['EBM'])}, "
      f"EEM={len(SUBGRAPH_RELATIONS['EEM'])}, 分配总计={total}")
print(f"   F4 层级边: {len(F4_RELATIONS)} 条（不进 F1 MetaPath 模板）")


✅ 关系清单：MPU=22, EBM=23, EEM=23, 分配总计=68
   F4 层级边: 10 条（不进 F1 MetaPath 模板）


In [ ]:
# ══════════════════════════════════════════════════════════════
# F1 验收：SUBGRAPH_RELATIONS 在 Neo4j 中的匹配情况
# ══════════════════════════════════════════════════════════════

import pandas as pd

rows = []
with neo4j_driver.session() as session:
    for sg, triples in SUBGRAPH_RELATIONS.items():
        for source, relation, target in triples:
            q = f"MATCH (s:{source})-[r:{relation}]->(t:{target}) RETURN count(*) AS c"
            count = session.run(q).single()["c"]
            rows.append({
                "subgraph": sg,
                "source": source,
                "relation": relation,
                "target": target,
                "count": count,
                "ok": count > 0,
            })

df = pd.DataFrame(rows)
ok = df["ok"].sum()
total = len(df)
zero = df[~df["ok"]]

print("=" * 60)
print(f"F1 关系模板验收: {ok}/{total} 条在 Neo4j 有实例 (count>0)")
print("=" * 60)

for sg in ["MPU", "EBM", "EEM"]:
    sub = df[df["subgraph"] == sg]
    sg_ok = sub["ok"].sum()
    print(f"  {sg}: {sg_ok}/{len(sub)} 有实例")

if len(zero):
    print(f"\n⚠️  无匹配实例 ({len(zero)} 条) — 可能 KG 未抽取到，或 label/关系名不一致:")
    for _, row in zero.iterrows():
        print(
            f"   [{row['subgraph']}] {row['source']}-[{row['relation']}]->{row['target']}"
        )
else:
    print("\n✅ 所有 F1 关系模板均在 Neo4j 中有至少 1 条实例")


## F1.2 定义构建metaPath函数

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd()
if not (_ROOT / "utilities").is_dir():
    raise FileNotFoundError(f"utilities 目录不存在: {_ROOT / 'utilities'}")
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from utilities.metapath_path_level import build_metapath_for_relation

print("✅ F1.2: build_metapath_for_relation（path_level=low）已加载")


## F1.3 构建metapath主程序

In [ ]:
from utilities.metapath_path_level import build_all_metapaths

def run_f1_build(driver):
    """F1: 全量 low MetaPath；任一条模板异常则中止。"""
    return build_all_metapaths(driver, SUBGRAPH_RELATIONS, PAGERANK_PROP)


print("✅ F1.3: run_f1_build 已定义")
print("   执行: summary = run_f1_build(neo4j_driver)")


In [ ]:
# F1 全量构建 low MetaPath（需先跑 Neo4j 连接 + SUBGRAPH_RELATIONS + F1.2/F1.3）
summary = run_f1_build(neo4j_driver)
summary


In [ ]:
from utilities.metapath_path_level import verify_metapath_path_level

def verify_metapath_creation(driver, *, require_mid: bool = False):
    """包装严格验收；F1 后 require_mid=False，F4 后 require_mid=True。"""
    return verify_metapath_path_level(driver, require_mid=require_mid)


print("✅ verify_metapath_creation / verify_metapath_path_level 已加载")
print("   F1 后: verify_metapath_creation(neo4j_driver, require_mid=False)")
print("   F4 后: verify_metapath_creation(neo4j_driver, require_mid=True)")


## F4 构建 mid MetaPath 与层级边

**判定规则（构建时确定）**

| path_level | 构建入口 | 含义 |
|------------|----------|------|
| low | F1 SUBGRAPH_RELATIONS | 原子 2-hop 路径 |
| mid | F4 Plan/容器聚合 | 每 Plan 或 SupportGraph/ScienceEvidence 实例 1 条 |

**maxPageRank（统一规则，low / mid 相同）**

```
maxPageRank(mp) = max( e.{sg}_pagerank
                     for (mp)-[:metaPathRelation]->(e) )
```

- `{sg}` 由 `mp.subgraph` 决定：`MPU`→`mpu_pagerank`，`EEM`→`eem_pagerank`，`EBM`→`ebm_pagerank`
- 只统计 **metaPathRelation 直接相连** 的基础节点（不沿 Plan/Part 等间接扩展）
- 每个相连节点必须有 E 段写回的 pagerank（NULL → raise，不用 0 掩盖）
- **low** 通常 |E|=2（source/target）；**mid** 通常 |E|=1（Plan/容器锚点）


**前置**：必须先完成 **E 段** GDS 写回。

**层级边**：hasDetailPath（mid→low）、detailOf（low→mid）

**链接**：同子图内 mid 锚点与 low 实体 **共享 FROM_CHUNK**（当前 KG 中 Step/part 节点 ID 与 F1 low 无交集）

**执行**：F1 完成后运行下方 cell；验收失败将 raise。


In [ ]:
from utilities.metapath_path_level import (
    build_mid_metapaths_for_plans,
    build_mid_metapaths_for_containers,
    link_mid_to_low,
    refresh_and_verify_metapath_max_pagerank,
    verify_metapath_path_level,
)

print("=" * 60)
print("F4: mid MetaPath + hasDetailPath / detailOf")
print("=" * 60)

mid_counters = {"MPU": 1, "EBM": 1, "EEM": 1}
mid_counters = build_mid_metapaths_for_plans(neo4j_driver, mid_counters)
mid_counters = build_mid_metapaths_for_containers(neo4j_driver, mid_counters)
link_stats = link_mid_to_low(neo4j_driver)
verify_metapath_path_level(neo4j_driver, require_mid=True)

print("\n" + "=" * 60)
print("F4.1: maxPageRank 全量刷新与统计（统一规则）")
print("=" * 60)
pr_report = refresh_and_verify_metapath_max_pagerank(neo4j_driver, pagerank_prop=PAGERANK_PROP)
pr_report


## F2 基于 metaPathText 生成 metaPathQuery（LLM）

- 处理 **全部** `:MetaPath`（含 `path_level` 为 `low` 与 `mid`）
- LLM 失败 **直接 raise**，不使用 metaPathText 兜底
- **前置**：F4 完成且验收通过


In [ ]:
import time
from neo4j_graphrag.llm import OpenAILLM

# ══════════════════════════════════════════════════════════════
# 专用 LLM 实例（temperature=0.0, max_tokens=200）
# ══════════════════════════════════════════════════════════════

llm_for_metapath = OpenAILLM(
    model_name="qwen-plus",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.environ.get("QWEN_API_KEY"),
    model_params={
        "temperature": 0.1,
        "max_tokens": 200,
    },
)

# ══════════════════════════════════════════════════════════════
# Few-Shot Prompt
# ══════════════════════════════════════════════════════════════

METAPATH_QUERY_PROMPT = """You are a scientific knowledge expert. 
Rewrite the following structured knowledge path into ONE natural English sentence 
that will be used for semantic search.

Requirements:
1. Keep all key entities (dataset names, claim names, method names) visible.
2. Preserve specific numerical values, chemical names, and technical terms.
3. Express the relationship between source and target clearly in natural language.
4. Use keywords that researchers would use to search for this knowledge.
5. Output ONE sentence only, no explanation, no prefix.
6. Length: 20-60 words.

Examples:

Input: [whu_DataSet: Hg_Rice_2024] Mercury concentration data in rice grain samples collected from Guizhou province -[mp_supports] The statistical analysis indicates a strong positive correlation -> [mp_Claim: Hg_methylation_sulfate] Mercury methylation is enhanced by sulfate-reducing bacteria under anaerobic conditions
Output: The Hg_Rice_2024 dataset showing mercury concentrations in Guizhou rice grain samples supports the claim that mercury methylation is enhanced by sulfate-reducing bacteria under anaerobic conditions.

Input: [whu_Bio_chemical_Experiment: CV_AFS_detection] Cold vapor atomic fluorescence spectrometry experiment -[whu_hasActivity] The experimental procedure involves sample digestion and mercury reduction -> [whu_BioChemicalActivityStep: sample_digestion] Acid digestion step using nitric acid and hydrogen peroxide
Output: The CV_AFS_detection experiment for mercury analysis includes an acid digestion step using nitric acid and hydrogen peroxide as part of its sample preparation activity.

Input: [whu_Specimen: rice_grain_sample] Rice grain samples collected from field trial -[p_plan_isOutputVarOf] Samples were obtained from the collection activity -> [whu_SpecimenCollection: field_sampling_2023] Field sampling campaign conducted in 2023 across multiple sites
Output: Rice grain samples obtained from the field_sampling_2023 collection campaign across multiple sites in 2023 serve as specimens for mercury analysis.

Now rewrite this knowledge path:

Input: {metapath_text}
Output:"""


def generate_metapath_query(metapath_text: str, retry: int = 2) -> str:
    """
    调用 LLM 基于 metaPathText 生成 metaPathQuery。
    失败时返回 metaPathText 作为 fallback。
    """
    if not metapath_text or len(metapath_text.strip()) < 10:
        return metapath_text
    
    prompt = METAPATH_QUERY_PROMPT.format(metapath_text=metapath_text)
    
    for attempt in range(retry + 1):
        try:
            response = llm_for_metapath.invoke(prompt)
            result = response.content.strip()
            
            # 清理
            result = result.strip('"\'')
            if result.lower().startswith(("output:", "rewrite:", "result:", "answer:")):
                result = result.split(":", 1)[1].strip()
            if "\n" in result:
                result = result.split("\n")[0].strip()
            
            # 质量校验
            if len(result) < 20:
                print(f"    ⚠️  结果过短 ({len(result)}字符)，重试...")
                continue
            if len(result) > 600:
                print(f"    ⚠️  结果过长 ({len(result)}字符)，截断")
                result = result[:600]
            
            return result
        
        except Exception as e:
            print(f"    ❌ LLM 调用失败（第 {attempt+1} 次）: {str(e)[:80]}")
            if attempt < retry:
                time.sleep(2)
    
    raise RuntimeError(f"LLM metaPathQuery 生成失败，已重试 {retry + 1} 次")


print("✅ Few-Shot metaPathQuery 生成函数已定义")
print(f"   LLM: qwen-plus, temperature=0.0, max_tokens=200")

In [ ]:
def batch_generate_metapath_query(
    driver,
    batch_size: int = 50,
    verbose: bool = True,
) -> dict:
    """
    批量为所有 metaPathQuery 为 NULL 的 MetaPath Node 生成 metaPathQuery。
    """
    # 查询待处理
    query_pending = """
    MATCH (mp:MetaPath)
    WHERE mp.metaPathQuery IS NULL
    RETURN mp.mp_id AS mp_id, mp.metaPathText AS metapath_text
    """
    
    with driver.session() as session:
        pending = session.run(query_pending).data()
    
    total = len(pending)
    if total == 0:
        print("  ℹ️  无待处理的 MetaPath")
        return {"total": 0, "success": 0, "failed": 0}
    
    print(f"\n{'=' * 60}")
    print(f"批量生成 metaPathQuery")
    print(f"{'=' * 60}")
    print(f"  待处理数量: {total}")
    print(f"  预估耗时: {total * 2 / 60:.1f} 分钟（按 2 秒/条）\n")
    
    stats = {"total": total, "success": 0, "failed": 0}
    start_time = time.time()
    
    update_query = """
    MATCH (mp:MetaPath {mp_id: $mp_id})
    SET mp.metaPathQuery = $metapath_query
    """
    
    with driver.session() as session:
        for idx, row in enumerate(pending, 1):
            mp_id = row["mp_id"]
            metapath_text = row["metapath_text"]
            
            if verbose:
                print(f"  [{idx}/{total}] {mp_id}")
                print(f"    原文: {metapath_text[:80]}...")
            
            metapath_query = generate_metapath_query(metapath_text)
            stats["success"] += 1
            
            if verbose:
                print(f"    重写: {metapath_query[:80]}...")
                print()
            
            session.run(update_query, mp_id=mp_id, metapath_query=metapath_query)
            
            # 进度展示
            if idx % batch_size == 0:
                elapsed = time.time() - start_time
                rate = idx / elapsed
                eta = (total - idx) / rate if rate > 0 else 0
                print(f"  ─── 进度: {idx}/{total} ({idx/total*100:.1f}%), "
                      f"速率: {rate:.1f}/s, 预计剩余: {eta/60:.1f} min ───\n")
    
    elapsed = time.time() - start_time
    print(f"\n{'=' * 60}")
    print(f"生成完成")
    print(f"{'=' * 60}")
    print(f"  总计: {stats['total']}")
    print(f"  成功: {stats['success']}")
    print(f"  失败: {stats['failed']}")
    print(f"  成功率: {stats['success']/stats['total']*100:.1f}%")
    print(f"  耗时: {elapsed:.1f}s ({elapsed/60:.1f} min)")
    print(f"  平均: {elapsed/stats['total']:.2f}s/条")
    if stats['failed']:
        raise RuntimeError(f"metaPathQuery 生成失败 {stats['failed']} 条")
    return stats


# ══════════════════════════════════════════════════════════════
# 直接执行全量生成（无 POC）
# ══════════════════════════════════════════════════════════════

stats = batch_generate_metapath_query(neo4j_driver, batch_size=50, verbose=True)



## F3 embedding + 向量/全文索引

- 对已有 `metaPathQuery` 的 MetaPath 写 `embedding`（BCE 768 维）
- 重建 `metapath_embedding_index`（向量）与 `metapath_fulltext_index`（metaPathText）
- **前置**：F2 完成


In [ ]:
import time
import numpy as np
from neo4j import GraphDatabase
from neo4j_graphrag.indexes import create_fulltext_index
from neo4j_graphrag.retrievers import HybridCypherRetriever
import pprint as p
import neo4j
import importlib
import os
from neo4j_graphrag.embeddings.sentence_transformers import SentenceTransformerEmbeddings
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.retrievers import Text2CypherRetriever
from neo4j_graphrag.generation import GraphRAG
def batch_generate_embeddings(
    driver,
    embed_model,
    batch_size: int = 32,
    verbose: bool = True,
) -> dict:
    """
    为所有 embedding 为 NULL 的 MetaPath Node 生成 embedding。
    
    基于 metaPathQuery 字段生成向量，写入 embedding 属性。
    """
    # 查询待处理
    query_pending = """
    MATCH (mp:MetaPath)
    WHERE mp.metaPathQuery IS NOT NULL AND mp.embedding IS NULL
    RETURN mp.mp_id AS mp_id, mp.metaPathQuery AS metapath_query
    """
    
    with driver.session() as session:
        pending = session.run(query_pending).data()
    
    total = len(pending)
    if total == 0:
        print("  ℹ️  无待处理的 MetaPath")
        return {"total": 0, "success": 0, "failed": 0}
    
    print(f"\n{'=' * 60}")
    print(f"批量生成 embedding")
    print(f"{'=' * 60}")
    print(f"  待处理数量: {total}")
    print(f"  Batch size: {batch_size}")
    print(f"  预估耗时: {total / 50:.1f} 分钟（按 50条/s）\n")
    
    stats = {"total": total, "success": 0, "failed": 0}
    start_time = time.time()
    
    update_query = """
    MATCH (mp:MetaPath {mp_id: $mp_id})
    SET mp.embedding = $embedding
    """
    
    # 分批处理
    with driver.session() as session:
        for batch_start in range(0, total, batch_size):
            batch = pending[batch_start:batch_start + batch_size]
            texts = [row["metapath_query"] for row in batch]
            
            try:
                # 批量生成 embedding（BCE 支持 batch 输入）
                embeddings = embed_model.embed_query(texts) if len(texts) == 1 \
                             else [embed_model.embed_query(t) for t in texts]
                
                # 写回 Neo4j
                for row, emb in zip(batch, embeddings):
                    if isinstance(emb, np.ndarray):
                        emb = emb.tolist()
                    session.run(
                        update_query,
                        mp_id=row["mp_id"],
                        embedding=emb,
                    )
                    stats["success"] += 1
                
                if verbose:
                    elapsed = time.time() - start_time
                    done = batch_start + len(batch)
                    rate = done / elapsed if elapsed > 0 else 0
                    eta = (total - done) / rate if rate > 0 else 0
                    print(f"  [{done}/{total}] ({done/total*100:.1f}%), "
                          f"速率: {rate:.1f}/s, 剩余: {eta/60:.1f} min")
            
            except Exception as e:
                print(f"  ❌ Batch 失败 ({batch_start}): {str(e)[:80]}")
                stats["failed"] += len(batch)
    
    elapsed = time.time() - start_time
    print(f"\n{'=' * 60}")
    print(f"Embedding 生成完成")
    print(f"{'=' * 60}")
    print(f"  总计: {stats['total']}")
    print(f"  成功: {stats['success']}")
    print(f"  失败: {stats['failed']}")
    print(f"  耗时: {elapsed:.1f}s ({elapsed/60:.1f} min)")
    
    return stats

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
neo4j_embed_model = SentenceTransformerEmbeddings(model=os.environ["LOCAL_MODEL_PATH_BCE"])
# 执行
stats = batch_generate_embeddings(neo4j_driver, neo4j_embed_model, batch_size=32)

构建 index

VECTOR_INDEX_NAME = "metapath_embedding_index"
FULLTEXT_INDEX_NAME = "metapath_fulltext_index"

In [ ]:
from neo4j_graphrag.indexes import (
    create_vector_index,
    create_fulltext_index,
    drop_index_if_exists,
)

# ══════════════════════════════════════════════════════════════
# Vector Index（基于 embedding 属性）
# ══════════════════════════════════════════════════════════════

VECTOR_INDEX_NAME = "metapath_embedding_index"
FULLTEXT_INDEX_NAME = "metapath_fulltext_index"

print("=" * 60)
print("建立 Vector Index")
print("=" * 60)

# 先删除已存在的索引（便于重建）
drop_index_if_exists(neo4j_driver, VECTOR_INDEX_NAME)

create_vector_index(
    driver=neo4j_driver,
    name=VECTOR_INDEX_NAME,
    label="MetaPath",
    embedding_property="embedding",
    dimensions=768,              # BCE embedding 维度
    similarity_fn="cosine",      # 余弦相似度
)

print(f"✅ Vector Index 创建成功: {VECTOR_INDEX_NAME}")
print(f"   标签: MetaPath, 属性: embedding, 维度: 768, 相似度: cosine\n")

# ══════════════════════════════════════════════════════════════
# Fulltext Index（基于 metaPathText 属性）
# ══════════════════════════════════════════════════════════════

print("=" * 60)
print("建立 Fulltext Index")
print("=" * 60)

drop_index_if_exists(neo4j_driver, FULLTEXT_INDEX_NAME)

create_fulltext_index(
    driver=neo4j_driver,
    name=FULLTEXT_INDEX_NAME,
    label="MetaPath",
    node_properties=["metaPathText"],
)

print(f"✅ Fulltext Index 创建成功: {FULLTEXT_INDEX_NAME}")
print(f"   标签: MetaPath, 属性: metaPathText\n")

# 附录：MetaPath 数据模型与构建结果总结

> 便于记忆与对照下游 [`3_0_2 Retevie.ipynb`](./3_0_2%20Retevie.ipynb) 多轮检索（`path_level` + `κ` drill_down/roll_up）。

---

## 一、两个正交维度

| 维度 | 取值 | 含义 |
|------|------|------|
| **subgraph** | `MPU` / `EBM` / `EEM` | 子图归属（与 `subgraph_mapping.json` 一致） |
| **path_level** | `low` / `mid` | 检索粒度：原子证据 vs 聚合概览 |

二者独立：`subgraph=MPU` 可同时有 `low` 与 `mid` MetaPath。

---

## 二、`:MetaPath` 节点属性（当前结构）

```
(:MetaPath {
  mp_id:         String,    // low: {SG}_{000001}；mid: {SG}_MID_{00001}
  metaPathText:  String,    // 结构化路径原文（F1/F4 构建）
  metaPathQuery: String,    // F2 LLM 改写，面向检索
  embedding:     List<Float>,// F3 BCE 768 维向量
  maxPageRank:   Float,     // max(e.{sg}_pagerank) for (mp)-[:metaPathRelation]->(e)
  subgraph:      String,    // MPU | EBM | EEM
  path_type:     String,    // 如 whu_DataSet-[mp_supports]->mp_Claim 或 mid_plan-*
  path_level:    String,    // "low" | "mid"（构建时写死，非检索推断）
  anchor_label:  String     // low: source_label；mid: Plan/容器 label
})
```

---

## 三、MetaPath 与 KG 实体的关系

F1 **low**（2-hop 原子路径）：

```
(mp:MetaPath {path_level:'low'})
  -[:metaPathRelation {position:1, relationText:...}]-> (source 实体)
  -[:metaPathRelation {position:2, relationText:null}]-> (target 实体)
```

F4 **mid**（聚合锚点，只连 1 个实体）：

```
(mp:MetaPath {path_level:'mid'})
  -[:metaPathRelation {position:1, relationText:null}]-> (Plan | SupportGraph | ScienceEvidence)
```

实体节点仍可通过 `[:FROM_CHUNK]->(:Chunk)` 关联原文段落。

---

## 四、MetaPath 之间的层级关系（论文 κ）

| 关系类型 | 方向 | 用途 |
|----------|------|------|
| **hasDetailPath** | `mid → low` | drill_down（下钻到细粒度证据） |
| **detailOf** | `low → mid` | roll_up（上卷到聚合概览） |

**链接规则（当前实现）**：同 `subgraph` 内，mid 锚点实体与 low 路径实体 **共享同一 Chunk**（`FROM_CHUNK` 共现）时建立边。

> **为何不用纯 Step/part 拓扑？** 诊断发现 F1 low 上的 Step/part **节点 ID** 与 KG 中 `isStepOfPlan` / `whu_hasPart` 所用节点 **无交集**（overlap=0）。若未来在 KG 构建阶段对齐节点 ID，可改回拓扑匹配。

**约束（验收）**：

- 仅允许 `mid → low` 的 `hasDetailPath`（禁止 mid→mid、low→low）
- 每个 mid 至少连 1 条 low（否则验收 `raise`）

---

## 五、low / mid 判定与构建入口

| path_level | 构建段 | 判定依据 |
|------------|--------|----------|
| **low** | **F1** `build_metapath_for_relation` | 模板 ∈ `SUBGRAPH_RELATIONS`（68 条）；**不含** `p_plan_isStepOfPlan`、`whu_hasPart` |
| **mid** | **F4** Plan/容器聚合 | 每个 Plan 或 SupportGraph/ScienceEvidence **实例** 1 条 mid |

代码集中在 [`utilities/metapath_path_level.py`](./utilities/metapath_path_level.py)。

---

## 六、执行顺序（本 Notebook）

```
E（PageRank）
  → F1 清单/验收
  → F1 low 全量构建
  → F4 mid + hasDetailPath/detailOf + 严格验收
  → F2 metaPathQuery（LLM，失败 raise，无 fallback）
  → F3 embedding + 向量/全文索引
```

---

## 七、最近一次全量构建结果（参考）

| 指标 | 数值 |
|------|------|
| **low 总计** | 1848（MPU 1182 / EBM 371 / EEM 295） |
| **mid 总计** | 176（MPU 156 / EBM 8 / EEM 12） |
| **hasDetailPath 边** | 4465 |
| **metaPathQuery** | 2024 / 2024 |
| **embedding** | 2024 / 2024 |
| **索引** | `metapath_embedding_index`（768 维 cosine）、`metapath_fulltext_index`（metaPathText） |

验收 Cypher：

```cypher
// 按 path_level 统计
MATCH (mp:MetaPath)
RETURN mp.subgraph, mp.path_level, count(*) ORDER BY 1, 2;

// 层级边
MATCH (mid:MetaPath {path_level:'mid'})-[h:hasDetailPath]->(low:MetaPath {path_level:'low'})
RETURN count(h);
```

---

## 八、与下游检索的对应（`3_0_2`）

| 对话操作 κ | 检索行为 |
|------------|----------|
| `first_turn` | 按 `subgraph + path_level`（默认 mid）向量/全文混合检索 |
| `drill_down` | 从 mid 锚点沿 `hasDetailPath` 取 low，不全库检索 |
| `roll_up` | 从 low 锚点沿 `detailOf` 取 mid |
| `sibling_nav` | 换 subgraph，path_level 不变 |

状态字段：`path_level`、`kappa`、`anchor_mp_ids`（见检索 Notebook G 段）。

---

## 九、辅助脚本

| 脚本 | 用途 |
|------|------|
| `utilities/metapath_path_level.py` | F1/F4 构建、连边、严格验收 |
| `utilities/run_metapath_pipeline.py` | 全流程（清理→F1→F4→F2→F3） |
| `utilities/run_metapath_f2_f3.py` | 仅续跑 F2/F3 |
| `utilities/check_metapath_status.py` | 快速查看 Neo4j 中 MetaPath 状态 |
